In [ ]:
import pandas as pd
import re


RUN_DATE = "20260811"

jd = pd.read_csv("../data/raw/jd_full_text_20260811.csv")
meta = pd.read_csv("../data/processed/jobs_cleaned_20260808.csv")

df = jd.merge(meta, on="id", how="left")
df["jd_lower"] = df["jd_text"].str.lower()
df["title_lower"] = df["title"].str.lower()
print(df.shape)

In [ ]:
SKILLS = {
    "cloud": ["aws", "s3", "ec2", "lambda", "sagemaker", "redshift",
              "athena", "azure", "gcp", "bigquery", "databricks",
              "snowflake", "terraform", "kubernetes", "docker"],
    "language": ["python", "r", "sql", "scala", "java", "c++",
                 "matlab", "sas", "julia", "fortran"],
    "library": ["pandas", "numpy", "scikit-learn", "pyspark", "spark",
                "airflow", "tensorflow", "pytorch", "keras", "git"],
    "bi": ["excel", "power bi", "tableau", "looker", "vba"],
    "method": ["machine learning", "deep learning", "nlp", "statistics",
               "a/b testing", "forecasting", "etl", "mlops",
               "numerical methods", "hpc", "parallel computing",
               "optimisation", "simulation", "rag", "vector database",
               "embeddings"],
}

all_skills = [s for group in SKILLS.values() for s in group]
print(len(all_skills))

df["jd_lower"] = df["jd_text"].str.lower()
print(df["jd_lower"].iloc[0][:200])

In [ ]:
naive = {}
for s in all_skills:
    naive[s] = df["jd_lower"].str.contains(s, regex=False).sum()

res = pd.Series(naive).sort_values(ascending=False)
print(res.head(20))

In [ ]:
import re

def build_pattern(skill):
    """把技能词转成带词边界的正则"""
    return r"\b" + re.escape(skill) + r"\b"

for s in ["r", "rag", "scala", "excel", "c++", "a/b testing", "power bi"]:
    print(f"{s:15s} {build_pattern(s)}")

In [ ]:
counts = {}
for s in all_skills:
    p = build_pattern(s)
    counts[s] = df["jd_lower"].str.contains(p, regex=True).sum()

res2 = pd.Series(counts).sort_values(ascending=False)
print(res2.head(25))

In [ ]:
def show_context(skill, n=8, width=60):
    """打印某技能词命中处的前后文"""
    p = build_pattern(skill)
    hits = df[df["jd_lower"].str.contains(p, regex=True)]
    print(f"=== {skill}  命中 {len(hits)} 条，抽 {min(n, len(hits))} 条 ===")
    for t in hits["jd_lower"].head(n):
        m = re.search(p, t)
        s = max(0, m.start() - width)
        e = min(len(t), m.end() + width)
        print("…" + t[s:e].replace("\n", " ") + "…")
    print()

for s in ["r", "excel", "git", "java"]:
    show_context(s)

In [ ]:
def build_pattern(skill):
    """带词边界的正则；对已知歧义词做特殊处理"""
    special = {
        # R：排除 r&d
        "r": r"\br\b(?!\s*&\s*d\b)",
        # excel：排除动词用法 "excel at/in/if/when"
        "excel": r"\bexcel\b(?!\s+(?:at|in|if|when)\b)",
        # git：纳入 github、gitlab
        "git": r"\bgit(?:hub|lab)?\b",
        # sql：纳入 postgresql、mysql、nosql 等变体
        "sql": r"\b(?:my|postgre|no|t-|pl/)?sql\b",
    }
    if skill in special:
        return special[skill]
    return r"\b" + re.escape(skill) + r"\b"

for s in ["r", "excel", "git", "sql", "python"]:
    print(f"{s:10s} {build_pattern(s)}")

In [ ]:
df["title_lower"] = df["title"].str.lower()

SENIORITY = [
    ("graduate", r"\b(graduate|intern|internship|placement|trainee|apprentice)\b"),
    ("junior",   r"\b(junior|entry.level|jr\.?)\b"),
    ("senior",   r"\b(senior|snr\.?|sr\.?)\b"),
    ("lead",     r"\b(lead|principal|staff|head\s+of|director|chief)\b"),
]

def get_seniority(t):
    for label, pat in SENIORITY:
        if re.search(pat, t):
            return label
    return "unspecified"

df["seniority"] = df["title_lower"].apply(get_seniority)
print(df["seniority"].value_counts())

In [ ]:
for lab in ["graduate", "junior", "lead"]:
    print(f"=== {lab} ===")
    print(df[df["seniority"] == lab]["title"].head(10).to_string(index=False))
    print()

In [ ]:
print(df[df["title_lower"].str.contains("junior|jr|entry", regex=True)]["title"].to_string(index=False))
print("---")
print(df[df["title_lower"].str.contains("manager", regex=True)]["title"].to_string(index=False))

In [ ]:
meta_all = pd.read_csv("../data/processed/jobs_cleaned_20260808.csv")
t = meta_all["title"].str.lower()

for pat in ["junior", "jr", "entry", "graduate", "trainee", "placement"]:
    n = t.str.contains(r"\b" + pat + r"\b", regex=True).sum()
    print(f"{pat:12s} {n:>4d}  ({n/len(meta_all)*100:.1f}%)")

In [ ]:
for s in all_skills:
    df["has_" + s] = df["jd_lower"].str.contains(build_pattern(s), regex=True)

skill_cols = ["has_" + s for s in all_skills]

grouped = df.groupby("seniority")[skill_cols].mean().T
grouped.columns = [f"{c}(n={(df['seniority']==c).sum()})" for c in grouped.columns]
grouped["overall"] = df[skill_cols].mean()

top = grouped.sort_values("overall", ascending=False).head(20)
print((top * 100).round(1).to_string())

In [ ]:
df.to_csv("../data/processed/jobs_with_skills_20260810.csv", index=False)
grouped.to_csv("../data/processed/skill_by_seniority_20260810.csv")
print(df.shape)